# 11 - Validation Curves

While Learning Curves change the *amount of data* on the X-axis, Validation Curves change a *single hyperparameter* on the X-axis to find the optimal model complexity.

## Concept Overview
We want to find the 'Sweet Spot' for a hyperparameter. Too low? Underfitting. Too high? Overfitting.

## Scikit-Learn Implementation
We will use `validation_curve` to test the `max_depth` parameter of a Random Forest.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import validation_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)
model = RandomForestClassifier(random_state=42)

# We will test max_depth from 1 to 20
param_range = np.arange(1, 21, 2)

train_scores, test_scores = validation_curve(
    model, X, y, param_name="max_depth", param_range=param_range,
    cv=5, scoring="accuracy", n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

## Visual Demonstration

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(param_range, train_mean, 'o-', color='blue', label='Training Score')
plt.plot(param_range, test_mean, 'o-', color='orange', label='Validation Score')

plt.fill_between(param_range, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
plt.fill_between(param_range, test_mean - test_std, test_mean + test_std, alpha=0.1, color='orange')

plt.title('Validation Curve for Random Forest')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.xticks(param_range)
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

## Interpretation
* **Left Side (Depth 1-3)**: Underfitting. Both scores are low.
* **Right Side (Depth 11+)**: Overfitting. The training score hits 1.0, but the validation score stops improving and might even decline slightly.
* **The Sweet Spot**: Depth 5 looks optimal. It gets maximum validation accuracy before the model starts memorizing noise.

## Evaluating Regularization
Let's test the `C` parameter in Logistic Regression. `C` is the inverse of regularization strength. Small `C` = strong regularization.

In [ ]:
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore') # Ignore convergence warnings for this demo

param_range_C = np.logspace(-4, 2, 7) # [0.0001, 0.001, ..., 100]

train_sc_C, test_sc_C = validation_curve(
    LogisticRegression(max_iter=1000), X, y, 
    param_name="C", param_range=param_range_C, cv=5
)

plt.figure(figsize=(8, 4))
plt.semilogx(param_range_C, np.mean(train_sc_C, axis=1), label='Train')
plt.semilogx(param_range_C, np.mean(test_sc_C, axis=1), label='Validation')
plt.xlabel('C (Inverse Regularization)')
plt.ylabel('Accuracy')
plt.title('Validation Curve: Logistic Regression C')
plt.legend()
plt.show()

## Industry Notes
Validation curves are great for understanding one parameter. But in reality, parameters interact. `max_depth` behaves differently depending on `min_samples_split`. To tune multiple parameters at once, we use Hyperparameter Tuning (next notebook).

## Exercises
1. Plot a validation curve for the `n_estimators` parameter in a Random Forest (try a range from 10 to 200). Does it overfit like `max_depth` does?
2. What is the difference between `param_range = np.arange(1, 10)` and `param_range = np.logspace(-3, 3, 7)`? When would you use each?